In [2]:
import pandas as pd
import os
import torch
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import os
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
from google.colab import drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**کنار گذاشتن درصدی از داده‌های آموزش به علت مساله جی‌پی‌یو با حفظ توزیع داده‌ها**

In [3]:
train_data = pd.read_csv('/content/drive/MyDrive/Poem Meter Dataset/train_samples.csv')
print(len(train_data))
new_weight_counts = train_data['metre'].value_counts()
threshold = 70000
high_count_weights = new_weight_counts[new_weight_counts > threshold].index.tolist()

high_count_weights_df = train_data[train_data['metre'].isin(high_count_weights)]
print("داده‌های پرتکرار:")
print(high_count_weights_df['metre'].value_counts().to_string())

# فیلتر کردن داده‌های پرتکرار و کاهش تعداد آن‌ها
reduced_samples = []
for weight in high_count_weights:

    high_weight_samples = train_data[train_data['metre'] == weight]
    # انتخاب تصادفی ۵۰ درصد داده‌ها
    sampled_high_weight = high_weight_samples.sample(frac=0.5, random_state=42)
    reduced_samples.append(sampled_high_weight)

low_count_samples = train_data[~train_data['metre'].isin(high_count_weights)]

balanced_train_samples = pd.concat([low_count_samples] + reduced_samples, axis=0)

output_dir = '/content/drive/MyDrive/poem-metre-datasets'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

balanced_train_samples.to_csv(f'{output_dir}/balanced_train_samples.csv', index=False)

749184
داده‌های پرتکرار:
metre
مفاعیلن مفاعیلن فعولن    149803
فعولن فعولن فعولن فعل    135237
فاعلاتن فاعلاتن فاعلن    116341
فعلاتن مفاعلن فعلن        73436


**بارگذاری داده‌ها و تمیزکردن دیتاست**

In [4]:
train_samples = pd.read_csv('/content/drive/MyDrive/poem-metre-datasets/balanced_train_samples.csv')
validation_samples = pd.read_csv('/content/drive/MyDrive/Poem Meter Dataset/validation_samples.csv')
test_samples = pd.read_csv('/content/drive/MyDrive/Poem Meter Dataset/test_samples.csv')

train_samples = train_samples.drop(columns=['poet_id', 'poem_id', 'v_order', 'metre_2'])
validation_samples = validation_samples.drop(columns=['poet_id', 'poem_id', 'v_order', 'metre_2'])
test_samples = test_samples.drop(columns=['poet_id', 'poem_id', 'v_order'])


print("Data loaded successfully.")
print(f"Train samples: {len(train_samples)}")
print(f"Validation samples: {len(validation_samples)}")
print(f"Test samples: {len(test_samples)}")


Data loaded successfully.
Train samples: 511775
Validation samples: 42545
Test samples: 42545


**آماده‌سازی داده‌های مصرع و وزن برای ورود به مدل ترنسفورمر**

In [5]:
tokenizer = AutoTokenizer.from_pretrained("HooshvareLab/bert-fa-base-uncased")
max_seq_length = 20
train_tokenized = tokenizer(
    train_samples['poem_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=max_seq_length,
    return_tensors="pt"
)

validation_tokenized = tokenizer(
    validation_samples['poem_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=max_seq_length,
    return_tensors="pt"
)

test_tokenized = tokenizer(
    test_samples['poem_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=max_seq_length,
    return_tensors="pt"
)

print("Train Tokenized Sample (input_ids):")
print(train_tokenized['input_ids'][:5])

print("\nValidation Tokenized Sample (input_ids):")
print(validation_tokenized['input_ids'][:5])

print("\nTest Tokenized Sample (input_ids):")
print(test_tokenized['input_ids'][:5])


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/440 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Train Tokenized Sample (input_ids):
tensor([[    2,  3051,  3206,  5200,  4694,  2806,  1379, 29088, 17744,     4,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [    2,  3051,  2985, 33930,  2791,  8800,  4990,  6847,  1350,     4,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [    2,  2938,  8887,  2882, 14013,  2791, 25670,  2015, 58740,  2003,
             4,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [    2, 17508,  3622, 76867,  2009,  2867,  3201,  2830,     4,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [    2, 66274,  3932,  1363,  6192, 16176,  3972,  6699,  2817,     4,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0]])

Validation Tokenized Sample (input_ids):
tensor([[    2,  2800,  2842,  2799, 16176,  1379,  2799,  2861,  2803, 11480,
         11480,     4,     0,     0,     0,    

In [6]:
all_meter_units = set()
for pattern in train_samples['metre']:
    all_meter_units.update(pattern.split())

# مپ کردن هر واحد وزنی به شناسه یکتا
meter_unit_to_id = {unit: idx + 1 for idx, unit in enumerate(all_meter_units)}

def map_meter_units_to_ids(meter_text, max_length):
    meter_units = meter_text.split()
    meter_ids = [meter_unit_to_id.get(unit, 0) for unit in meter_units]
    if len(meter_ids) < max_length:
        meter_ids += [0] * (max_length - len(meter_ids))  # پد
    else:
        meter_ids = meter_ids[:max_length]
    return meter_ids

train_meter_ids = [map_meter_units_to_ids(meter, max_seq_length) for meter in train_samples['metre']]
validation_meter_ids = [map_meter_units_to_ids(meter, max_seq_length) for meter in validation_samples['metre']]

# تبدیل به تنسور
train_meter_ids_tensor = torch.tensor(train_meter_ids)
validation_meter_ids_tensor = torch.tensor(validation_meter_ids)

print("Sample of token-by-token mapped meter IDs for training:")
print(train_meter_ids_tensor[:5])


Sample of token-by-token mapped meter IDs for training:
tensor([[12, 10, 12,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0],
        [12, 10, 12,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0],
        [10, 10, 10,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0],
        [12, 10, 12,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0],
        [12, 10, 12,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0]])


In [7]:
start_token = 0
train_decoder_input_ids = torch.cat([
    torch.full((train_meter_ids_tensor.size(0), 1), start_token, dtype=torch.long),
    train_meter_ids_tensor[:, :-1]
], dim=1)

validation_decoder_input_ids = torch.cat([
    torch.full((validation_meter_ids_tensor.size(0), 1), start_token, dtype=torch.long),
    validation_meter_ids_tensor[:, :-1]
], dim=1)

print("Sample of shifted meter IDs for Decoder Input (training):")
print(train_decoder_input_ids[:5])

train_dataset = TensorDataset(
    train_tokenized['input_ids'],
    train_tokenized['attention_mask'],
    train_decoder_input_ids,
    train_meter_ids_tensor  # لیبل
)

validation_dataset = TensorDataset(
    validation_tokenized['input_ids'],
    validation_tokenized['attention_mask'],
    validation_decoder_input_ids,
    validation_meter_ids_tensor  # لیبل)
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size)

for batch in train_loader:
    input_ids_batch, attention_mask_batch, decoder_input_ids_batch, labels_batch = batch
    print("Sample Input IDs Batch:", input_ids_batch)
    print("Sample Attention Mask Batch:", attention_mask_batch)
    print("Sample Decoder Input IDs Batch:", decoder_input_ids_batch)
    print("Sample Labels Batch:", labels_batch)
    break


Sample of shifted meter IDs for Decoder Input (training):
tensor([[ 0, 12, 10, 12,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0],
        [ 0, 12, 10, 12,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0],
        [ 0, 10, 10, 10,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0],
        [ 0, 12, 10, 12,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0],
        [ 0, 12, 10, 12,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0]])
Sample Input IDs Batch: tensor([[    2,  3360,  4572,  2800,  4266,  2842,  2789,  5419,  4361,  2786,
          4598,     4,     0,     0,     0,     0,     0,     0,     0,     0],
        [    2,  2791,  2802, 41964,  2009, 18739, 28728,     4,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [    2,  3534,  2010,  3550,  4349,  2029, 18575,  2812,  1379,  3491,
             4,     0,   

# **مدل ترنسفورمر**

In [8]:
class MeterPredictionTransformer(nn.Module):
    def __init__(self, vocab_size, num_classes, hidden_dim, num_heads, num_layers, dropout):
        super(MeterPredictionTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        # مدل ترنسفورمر با دیکودر و انکودر
        self.transformer = nn.Transformer(
            d_model=hidden_dim,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        # لایه خروجی برای پیش‌بینی کلاس وزن عروضی در هر موقعیت توکن
        self.fc_out = nn.Linear(hidden_dim, num_classes)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_key_padding_mask=None, tgt_key_padding_mask=None):

        src_embedded = self.embedding(src)  # تعبیه‌های ورودی کدگذار
        tgt_embedded = self.embedding(tgt)  # تعبیه‌های ورودی کدگشا

        transformer_out = self.transformer(
            src=src_embedded,
            tgt=tgt_embedded,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        # لایه خروجی طبقه‌بندی هر موقعیت به یک شناسه وزن عروضی
        output = self.fc_out(transformer_out)
        return output

# پارامترهای مدل
vocab_size = tokenizer.vocab_size
hidden_dim = 128
num_heads = 8
num_layers = 4
dropout = 0.1
num_classes = len(meter_unit_to_id) + 1

model = MeterPredictionTransformer(vocab_size, num_classes, hidden_dim, num_heads, num_layers, dropout)
print(model)


MeterPredictionTransformer(
  (embedding): Embedding(100000, 128)
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): TransformerDecoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerDecode

# **آموزش و ارزیابی مدل**

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

checkpoint_dir = "/content/drive/MyDrive/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

def train(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in tqdm(train_loader, desc="Training"):
        input_ids, attention_mask, decoder_input_ids, labels = [item.to(device) for item in batch]

        optimizer.zero_grad()
        outputs = model(input_ids, decoder_input_ids)

        outputs = outputs.view(-1, outputs.shape[-1])
        labels = labels.view(-1)

        loss = criterion(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(train_loader)
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=1)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=1)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=1)

    return avg_loss, precision, recall, f1

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            input_ids, attention_mask, decoder_input_ids, labels = [item.to(device) for item in batch]
            outputs = model(input_ids, decoder_input_ids)

            outputs = outputs.view(-1, outputs.shape[-1])
            labels = labels.view(-1)

            loss = criterion(outputs, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(val_loader)
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=1)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=1)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=1)

    return avg_loss, precision, recall, f1

# چک پوینت
def load_checkpoint(model, optimizer, checkpoint_dir):
    checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint_")]
    if checkpoint_files:
        latest_checkpoint = sorted(checkpoint_files, reverse=True)[0]
        checkpoint = torch.load(os.path.join(checkpoint_dir, latest_checkpoint), map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_f1 = checkpoint['best_val_f1']
        print(f"Resumed from checkpoint at epoch {start_epoch} with best F1 Score: {best_val_f1:.4f}")
        return start_epoch, best_val_f1
    else:
        print("No checkpoint found, starting from scratch.")
        return 0, 0.0

def save_checkpoint(epoch, model, optimizer, best_val_f1, checkpoint_dir):
    checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch+1}.pth")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_f1': best_val_f1
    }, checkpoint_path)
    print(f"Checkpoint saved at epoch {epoch + 1} with best F1 Score: {best_val_f1:.4f}")

start_epoch, best_val_f1 = load_checkpoint(model, optimizer, checkpoint_dir)
num_epochs = 10
best_model_path = "/content/drive/MyDrive/best_meter_prediction_model.pth"

for epoch in range(start_epoch, num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss, train_precision, train_recall, train_f1 = train(model, train_loader, optimizer, criterion, device)
    print(f"Training Loss: {train_loss:.4f}, Precision: {train_precision:.4f}, Recall: {train_recall:.4f}, F1 Score: {train_f1:.4f}")

    val_loss, val_precision, val_recall, val_f1 = evaluate(model, validation_loader, criterion, device)
    print(f"Validation Loss: {val_loss:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1 Score: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"Best model saved with Validation F1 Score: {best_val_f1:.4f}")
    save_checkpoint(epoch, model, optimizer, best_val_f1, checkpoint_dir)

print("Training completed.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<ipython-input-10-099b2fc7bd16>:85: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(os.path.join(checkpoint_dir, latest_checkpoint), map_location=devic

Resumed from checkpoint at epoch 1 with best F1 Score: 0.3184

Epoch 2/10


Training: 100%|██████████| 15993/15993 [09:17<00:00, 28.71it/s]


Training Loss: 0.3274, Precision: 0.3361, Recall: 0.7458, F1 Score: 0.3318


...

Validation Loss: 0.3315, Precision: 0.3671, Recall: 0.7488, F1 Score: 0.3203
Checkpoint saved at epoch 9 with best F1 Score: 0.3288

Epoch 10/10


Training: 100%|██████████| 15993/15993 [09:15<00:00, 28.79it/s]


Training Loss: 0.3111, Precision: 0.3409, Recall: 0.7477, F1 Score: 0.3295


Evaluating: 100%|██████████| 1330/1330 [00:09<00:00, 145.87it/s]


Validation Loss: 0.3302, Precision: 0.3365, Recall: 0.7744, F1 Score: 0.3280
Checkpoint saved at epoch 10 with best F1 Score: 0.3288
Training completed.


## **پیش بینی وزن برای داده‌های تست با جستجوی گریدی**

In [15]:

def predict(model, test_loader, device, max_seq_length=4):
    all_preds = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            input_ids, attention_mask, decoder_input_ids = [item.to(device) for item in batch]

            outputs = model(input_ids, decoder_input_ids)
            preds = torch.argmax(outputs[:, :max_seq_length, :], dim=-1)
            all_preds.extend(preds.cpu().numpy())

    return all_preds

test_decoder_input_ids = torch.cat([
    torch.full((test_tokenized['input_ids'].size(0), 1), start_token, dtype=torch.long),
    test_tokenized['input_ids'][:, :-1]
], dim=1)
#تنسور
test_dataset = TensorDataset(
    test_tokenized['input_ids'],
    test_tokenized['attention_mask'],
    test_decoder_input_ids
)

# دیتالودر
test_loader = DataLoader(test_dataset, batch_size=batch_size)

test_predictions = predict(model, test_loader, device, max_seq_length=max_seq_length)

# نگاشت شناسه به الگوی وزن اصلی
id_to_meter_unit = {v: k for k, v in meter_unit_to_id.items()}
predicted_meter_patterns = []

for pred_seq in test_predictions:
    mapped_sequence = [id_to_meter_unit.get(id, "") for id in pred_seq if id != 0]
    meter_pattern = " ".join(mapped_sequence)
    predicted_meter_patterns.append(meter_pattern)

test_samples['predicted_metre'] = predicted_meter_patterns
test_samples[['poem_text', 'predicted_metre']].to_csv('/content/predicted_meter_pattern.csv', index=False)




Predicting: 100%|██████████| 1330/1330 [00:09<00:00, 140.01it/s]


In [16]:
test_samples[['poem_text', 'predicted_metre']]

,poem_text,predicted_metre
0,گر درخور عشق آید خرم چو دمشق آید,مفعول مستفعلن فعل فعولن
1,ای صدر جهان جهان ندارد چو تویی,مفعول مفعول فعلن
2,شوی بی‌گزند از بد بدگمان,مفعول فعلن فاعلن
3,گه‌ کلیمی سازد از موسی و در دستش‌ کند,مفعول مفعول مفاعلن مفاعلن
4,بسی گفتی و در آخر رسیدی,مفعول فعلن فعلن
...,...,...
42540,ز درش به روز من ار چه دور همی روم,مفعول مستفعلن فاعلاتن مفاعیلن
42541,رخ او گلفشان شود نظرم گلستان شود,مفعول مستفعلن فعلن مفاعیلن
42542,بسر تو کین دل‌خسته را به نسیم خود خبری کنی,مفعول مفاعیلن مفاعیلن مفاعیلن
42543,آزاده نژاد از درم خرید,مفعول فعلن فعلن


## **پیش بینی وزن برای داده‌های تست با جستجوی بیم سرچ**

In [18]:
beam_size = 2
def predict_beam_search(model, test_loader, device, max_seq_length=4, beam_size=2):
    all_preds = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting with Beam Search", disable=True):

            input_ids, attention_mask, decoder_input_ids = [item.to(device) for item in batch]

            logits = model(input_ids, decoder_input_ids)
            beams = [([], 0.0)]

            for step in range(max_seq_length):
                new_beams = []

                for seq, log_prob in beams:
                    step_logits = logits[:, step, :].squeeze(0)
                    step_probs = torch.softmax(step_logits, dim=-1)

                    # انتخاب بالاترین احتمال به اندازه بیم سایز
                    top_probs, top_indices = torch.topk(step_probs, beam_size)

                    for i in range(beam_size):
                        new_seq = seq + [top_indices[i].item()]
                        new_log_prob = log_prob + torch.log(top_probs[i]).item()
                        new_beams.append((new_seq, new_log_prob))

                # نگهداری بهترین توالی بر اساس احتمال لگاریتمی
                beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

            # انتخاب بهترین توالی بیم
            best_sequence = beams[0][0]
            all_preds.append(best_sequence)

    return all_preds

test_decoder_input_ids = torch.cat([
    torch.full((test_tokenized['input_ids'].size(0), 1), start_token, dtype=torch.long),
    test_tokenized['input_ids'][:, :-1]
], dim=1)

test_dataset = TensorDataset(
    test_tokenized['input_ids'],
    test_tokenized['attention_mask'],
    test_decoder_input_ids
)

test_loader = DataLoader(test_dataset, batch_size=1)

test_predictions = predict_beam_search(model, test_loader, device, max_seq_length=max_seq_length, beam_size=beam_size)

id_to_meter_unit = {v: k for k, v in meter_unit_to_id.items()}
predicted_meter_patterns = []

for pred_seq in test_predictions:
    mapped_sequence = [id_to_meter_unit.get(id, "") for id in pred_seq if id != 0]
    meter_pattern = " ".join(mapped_sequence)
    predicted_meter_patterns.append(meter_pattern)

test_samples['predicted_metre'] = predicted_meter_patterns
test_samples[['poem_text', 'predicted_metre']].to_csv('/content/predicted_meter_pattern_beam_search.csv', index=False)


In [19]:
test_samples[['poem_text', 'predicted_metre']]

,poem_text,predicted_metre
0,گر درخور عشق آید خرم چو دمشق آید,مفعول مستفعلن فعل فعولن
1,ای صدر جهان جهان ندارد چو تویی,مفعول مفعول فعلن فعلن
2,شوی بی‌گزند از بد بدگمان,مفعول فعلن فاعلن فعلن
3,گه‌ کلیمی سازد از موسی و در دستش‌ کند,مفعول مفعول مفاعلن مفاعلن
4,بسی گفتی و در آخر رسیدی,مفعول فعلن فعلن فعلن
...,...,...
42540,ز درش به روز من ار چه دور همی روم,مفعول مستفعلن فاعلاتن مفاعیلن
42541,رخ او گلفشان شود نظرم گلستان شود,مفعول مستفعلن فعلن مفاعیلن
42542,بسر تو کین دل‌خسته را به نسیم خود خبری کنی,مفعول مفاعیلن مفاعیلن مفاعیلن
42543,آزاده نژاد از درم خرید,مفعول فعلن فعلن فعلن
